# organic_ratio — entry notebook


## 1. Git sync

Подтянуть актуальный `main` из `Anton-Filimoncev-azur/organic_ratio` поверх локальной копии. Требует `GIT_PAT` в `.env`.

In [1]:
import os
import subprocess
import base64
from pathlib import Path
from dotenv import load_dotenv

PROJECT_ROOT = "/home/jovyan/KEDRO/organic_ratio"
BRANCH = "main"
ORG = "Anton-Filimoncev-azur"
REPO = "organic_ratio"
USER = "Anton-Filimoncev-azur"

os.chdir(PROJECT_ROOT)
print("PWD:", Path().resolve())

load_dotenv()
TOKEN = os.getenv("GIT_PAT")
if not TOKEN:
    raise ValueError("GIT_PAT not found in environment")

auth = base64.b64encode(f"{USER}:{TOKEN}".encode()).decode()

subprocess.run(["git", "rebase", "--abort"], stderr=subprocess.DEVNULL)
subprocess.run(["git", "merge", "--abort"], stderr=subprocess.DEVNULL)
subprocess.run(["git", "reset", "--hard"], check=True)
subprocess.run(["git", "clean", "-fd"], check=True)

subprocess.run(
    [
        "git",
        "-c",
        f"http.extraheader=Authorization: Basic {auth}",
        "fetch",
        f"https://github.com/{ORG}/{REPO}.git",
        BRANCH,
    ],
    check=True,
)
subprocess.run(["git", "reset", "--hard", "FETCH_HEAD"], check=True)

print("Repository synced successfully.")

PWD: /home/jovyan/KEDRO/organic_ratio
HEAD is now at 099c228  work
HEAD is now at e8ea631  work
Repository synced successfully.


From https://github.com/Anton-Filimoncev-azur/organic_ratio
 * branch            main       -> FETCH_HEAD


## 2. Env

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

## 3. Ingestion (S3 → data/raw/partition/)

In [3]:
# %run run.py

## 4. Per-source preprocessing (raw → data/features/partition/)

In [4]:
# import gc
# %run run_preprocessing.py
# gc.collect()

## 5. Cohort aggregation

`run_cohort_aggregation.py` — мерджит per-source user-grain фичи и роллапит до cohort-grain по ключам из `cohort.keys`. Добавляет `cohort_size` и `n_calendar_days`.

In [5]:
# %run run_cohort_aggregation.py

## 6. Target + features build (model-ready train / test)

`run_target_build.py` за один проход:
1. Считает target (`organic_share`, `total_installs`, `organic_installs`) на грануляции `cohort.keys − media_source`.
2. Реагрегирует все user-grain фичи на ту же грануляцию (политика SUM/MEAN из `aggregate_to_cohort`).
3. Джойнит target + features, пишет:
   - `data/features/targets/targets.parquet` (полный + колонка `split`)
   - `data/train/targets_train.parquet` (split == train, без `split`)
   - `data/test/targets_test.parquet` (split == test, без `split`)

После этого `targets_train.parquet` / `targets_test.parquet` — это уже готовые для модели таблицы.

In [6]:
# import gc
# %run run_target_build.py
# gc.collect()

## 7. Clean (filter small cohorts)

`run_clean.py` — отрезает когорты, где `total_installs < cleaning.min_total_installs` (значение в `parameters.yml`). Пишет `targets_train_clean.parquet` и `targets_test_clean.parquet`.

In [7]:
# import gc
# %run run_clean.py
# gc.collect()

## 8. Baseline (weighted Ridge on logit)

`run_baseline.py` — линейный baseline:
- target: `logit(organic_share)`
- `sklearn.Ridge(alpha=1.0)` с `sample_weight = total_installs`
- preprocessing: numeric → median impute + StandardScaler; `platform` → one-hot; `country_code` → weighted target-encoding (mean от train); `install_date` → drop
- метрики (weighted/unweighted RMSE, MAE, R²) train + test
- top-20 коэффициентов
- сохраняет `data/predictions/baseline_{train,test}.parquet` и `data/plots/baseline_{calibration,coefficients}.png`

In [8]:
# %run run_baseline.py

## 9. PyMC hierarchical model

`run_train.py` — иерархическая байесовская модель:

```
organic_installs[i] ~ Binomial(total_installs[i], p[i])
logit(p[i]) = α + X[i]·β + u_country[c] + u_platform[plat]
u_country  ~ Normal(0, σ_country),  σ_country ~ HalfNormal(1)
u_platform ~ Normal(0, σ_platform), σ_platform ~ HalfNormal(1)
```

Binomial-правдоподобие учитывает размер когорты — `total_installs` есть в самой функции потерь, отдельных весов не нужно.

Параметры сэмплирования в `modeling.pymc` (`parameters.yml`):
- `draws: 500`, `tune: 500`, `chains: 2`, `target_accept: 0.9`
- `nuts_sampler: pymc` (можно `numpyro` для скорости, нужен `pip install numpyro jax`)

Артефакты: `data/models/pymc/{trace.nc, prep.pkl}`, `data/predictions/pymc_{train,test}.parquet`, `data/plots/pymc_{calibration,beta}.png`.

In [9]:
# %run run_train.py

## 10. MMM data prep

`run_mmm_data.py` собирает MMM-панель `platform × country × install_date`:
- `organic_installs` (target halo-модели), `total_installs`
- `spend_<top10_sources>`, `spend_other_paid`
- `dow_0..dow_6` (dayofweek)
- `geo` = `<platform>_<country>` (dim для иерархии)

Параметры в `modeling.mmm` (`parameters.yml`): `top_n_channels`, `min_country_installs`, `adstock_l_max`, `saturation`, `geo_dim`, `yearly_seasonality`.

Артефакт: `data/features/mmm/mmm_panel.parquet`.

In [10]:
%run run_mmm_data.py

Reading installs (features): data/features/partition/installs.parquet
Reading costs (raw):         data/raw/partition/costs.parquet
  top-10 channels: ['applovin_int', 'googleadwords_int', 'facebook ads', 'ironsource_int', 'unityads_int', 'mintegral_int', 'moloco_int', 'tiktokglobal_int', 'apple search ads', 'bytedanceglobal_int']

Panel shape: (47595, 24)
Date range:  2024-11-01 00:00:00  →  2025-05-31 00:00:00
Geos:        250  unique (platform × country)
Channels:    10 + other_paid

organic_installs summary:
shape: (9, 2)
┌────────────┬───────────┐
│ statistic  ┆ value     │
│ ---        ┆ ---       │
│ str        ┆ f64       │
╞════════════╪═══════════╡
│ count      ┆ 47595.0   │
│ null_count ┆ 0.0       │
│ mean       ┆ 25.994684 │
│ std        ┆ 85.733472 │
│ min        ┆ 0.0       │
│ 25%        ┆ 2.0       │
│ 50%        ┆ 5.0       │
│ 75%        ┆ 18.0      │
│ max        ┆ 2655.0    │
└────────────┴───────────┘

spend per channel (sum across panel):
  spend_apple search ads

## 11. MMM training (halo attribution)

`run_mmm_train.py` обучает halo-MMM:

```
organic_installs[g, t] = baseline[g] + Σ_source saturation(adstock(spend_source[g, t]))
                       + Σ_dow control_dow * dow_d[t]
```

`pymc-marketing.MMM` с:
- `GeometricAdstock(l_max=7)`
- `LogisticSaturation` (можно `michaelis_menten` / `hill`)
- `dims=("geo",)` — per-(platform × country) коэффициенты

Артефакты:
- `data/models/mmm/mmm.nc` — сохранённая MMM
- `data/predictions/mmm_train.parquet`
- `data/plots/mmm_{contribution,saturation,channel_share}.png`

In [11]:
%run run_mmm_train.py

Adding SRC_PATH: /home/jovyan/KEDRO/organic_ratio/src


WARNING (pytensor.configdefaults): g++ not detected!  PyTensor will be unable to compile C-implementations and will default to Python. Performance may be severely degraded. To remove this warning, set PyTensor flags cxx to an empty string.


[jax] default_backend: gpu
[jax] devices: [CudaDevice(id=0)]
Loading panel: data/features/mmm/mmm_panel.parquet
Panel: (47595, 24), geos=250, dates=2024-11-01 → 2025-05-31
Target:       organic_installs
Channels (11): ['spend_apple search ads', 'spend_applovin_int', 'spend_other_paid', 'spend_googleadwords_int', 'spend_unityads_int', 'spend_bytedanceglobal_int', 'spend_facebook ads', 'spend_mintegral_int', 'spend_ironsource_int', 'spend_moloco_int', 'spend_tiktokglobal_int']
Controls (7): ['dow_0', 'dow_1', 'dow_2', 'dow_3', 'dow_4', 'dow_5', 'dow_6']

Aggregating panel: 47595 rows → 1 row per date
Time-series: 212 days (2024-11-01 → 2025-05-31)

Fitting MMM: draws=500, tune=1500, chains=2, target_accept=0.9, sampler=numpyro, chain_method=vectorized


/home/jovyan/.local/lib/python3.11/site-packages/pydantic/_internal/_validate_call.py:136: FutureWarning: 
            The MMM class is deprecated and will be removed in a future version (in version 0.20.0).
            Please use the multidimensional MMM class instead.
            That is, `from pymc_marketing.mmm.multidimensional import MMM`.
            All our documentation has been updated to reflect this change.
            Refer to the migration guide for more details: https://www.pymc-marketing.io/en/latest/notebooks/mmm/mmm_migration_guide.html
            
  res = self.__pydantic_validator__.validate_python(pydantic_core.ArgsKwargs(args, kwargs))
sample: 100%|██████████| 2000/2000 [02:59<00:00, 11.13it/s]
There was 1 divergence after tuning. Increase `target_accept` or reparameterize.
We recommend running at least 4 chains for robust computation of convergence diagnostics
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See 

Output()


Saved MMM: data/models/mmm/mmm.nc

Computing posterior predictive (in-sample)...


TypeError: ('XTensorType{dtype=\'int8\', shape=(None, None), dims=(\'date\', \'control\')} cannot store a value of dtype int32 without risking loss of precision. If you do not mind this loss, you can: 1) explicitly cast your data to int8, or 2) set "allow_input_downcast=True" when calling "function". Value: "array([[0, 0, 0, ..., 1, 0, 0],\n       [0, 0, 0, ..., 0, 1, 0],\n       [0, 0, 0, ..., 0, 0, 1],\n       ...,\n       [0, 0, 0, ..., 0, 0, 0],\n       [0, 0, 0, ..., 1, 0, 0],\n       [0, 0, 0, ..., 0, 1, 0]], shape=(212, 7), dtype=int32)"', 'Container name "control_data"')

## (опц.) Пакетный запуск экспериментов через CONFIG_OVERRIDE_PATH

In [12]:
# import os
# from pathlib import Path
# from omegaconf import OmegaConf
#
# SWEEP_PATH = Path("conf/batch_training/sweep.yml")
# TMP_DIR = Path("conf/batch_training/.tmp")
#
# sweep_cfg = OmegaConf.load(SWEEP_PATH)
# TMP_DIR.mkdir(parents=True, exist_ok=True)
#
# for exp in sweep_cfg.experiments:
#     run_name = exp.name
#     override_path = TMP_DIR / f"{run_name}.yml"
#     OmegaConf.save(config=exp.overrides, f=override_path)
#     os.environ["CONFIG_OVERRIDE_PATH"] = str(override_path)
#     try:
#         %run run_train.py
#         %run run_eval.py
#     finally:
#         os.environ.pop("CONFIG_OVERRIDE_PATH", None)